# T34 - 债券新发行

## 第5章：核心逻辑实现

---

### 本章概述

本章实现债券新发行系统的核心业务逻辑，包括：
1. 债券到期数据采集主流程
2. 债券新发行数据采集主流程
3. 数据去重逻辑
4. 定时任务调度

### 导入依赖

In [ ]:
# 标准库
import os
import sys
import datetime
from datetime import datetime, timedelta
import logging
import warnings
from typing import Optional, List, Dict, Any

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text, create_engine, inspect
import sqlalchemy.pool as pool

# 忽略警告
warnings.filterwarnings('ignore')

print("依赖库导入成功")

### 配置和连接初始化

In [ ]:
# 日志配置
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

def log_progress(message):
    """记录处理进度"""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{current_time}] {message}")
    logger.info(message)

# 数据库配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'yq')

# 创建数据库连接
def get_db_engine():
    connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
    return create_engine(connection_string, poolclass=pool.NullPool, pool_recycle=3600, echo=False)

print("配置和连接初始化完成")

### 核心工具函数

In [ ]:
def generate_column_mapping(columns):
    """生成列名映射"""
    return {col: f"col_{i}" for i, col in enumerate(columns)}


def change_column_names(table_name, column_mapping, engine, to_english=True):
    """修改表列名"""
    with engine.begin() as connection:
        for original_name, new_name in column_mapping.items():
            if to_english:
                connection.execute(text(
                    f"ALTER TABLE `{table_name}` CHANGE `{original_name}` `{new_name}` VARCHAR(255);"
                ))
            else:
                connection.execute(text(
                    f"ALTER TABLE `{table_name}` CHANGE `{new_name}` `{original_name}` VARCHAR(255);"
                ))


def insert_data_to_db(df, table_name, engine):
    """
    将数据插入数据库（支持UPSERT）
    
    Args:
        df: 待插入的数据DataFrame
        table_name: 目标表名
        engine: 数据库引擎
    """
    if df is None or df.empty:
        log_progress(f"无数据需要插入到 {table_name}")
        return 0
    
    # 数据预处理
    df = df.copy()
    df = df.replace({pd.NA: None, pd.NaT: None, float('nan'): None})
    
    # 生成列名映射
    column_mapping = generate_column_mapping(df.columns)
    df = df.rename(columns=column_mapping)
    
    inspector = inspect(engine)
    table_exists = inspector.has_table(table_name)
    
    if table_exists:
        # 获取现有表的列名
        existing_columns = inspector.get_columns(table_name)
        existing_columns_names = [col['name'] for col in existing_columns]
        df_columns = df.columns.tolist()

        # 检查并添加缺失的列
        for col in df_columns:
            if col not in existing_columns_names:
                col_type = "FLOAT" if pd.api.types.is_numeric_dtype(df[col]) else "VARCHAR(255)"
                with engine.begin() as connection:
                    connection.execute(text(f"ALTER TABLE `{table_name}` ADD COLUMN `{col}` {col_type};"))
                log_progress(f"添加新列: {col} ({col_type})")

        # 构建UPSERT语句
        insert_columns = ', '.join([f"`{col}`" for col in df_columns])
        update_columns = ', '.join([f"`{col}` = VALUES(`{col}`)" for col in df_columns])
        value_placeholders = ', '.join([f":{col}" for col in df_columns])

        insert_query = text(f"""
        INSERT INTO `{table_name}` ({insert_columns})
        VALUES ({value_placeholders})
        ON DUPLICATE KEY UPDATE {update_columns};
        """)

        # 执行插入
        inserted_count = 0
        with engine.begin() as connection:
            for _, row in df.iterrows():
                try:
                    params = {col: row[col] for col in df_columns}
                    connection.execute(insert_query, params)
                    inserted_count += 1
                except Exception as e:
                    log_progress(f"插入失败: {e}")
        
        log_progress(f"成功插入 {inserted_count} 条记录到 {table_name}")
        return inserted_count
    
    return 0

print("核心工具函数定义完成")

### 债券到期数据采集类

In [ ]:
class BondMaturityCollector:
    """债券到期数据采集器"""
    
    # 债券类型配置
    BOND_TYPES = [
        ('governmentbonds', '国债', '否'),
        ('centralbankbills', '央行票据', '否'),
        ('cds', '存单', '否'),
        ('commercialbankbonds', '普通金融债', '否'),
        ('policybankbonds', '政策银行债', '否'),
        ('commercialbanksubordinatedbonds', '商业银行次级债券', '否'),
        ('insurancecompanybonds', '保险债', '否'),
        ('corporatebondsofsecuritiescompany', '证券公司债', '否'),
        ('securitiescompanycps', '证券公司短融债', '否'),
        ('otherfinancialagencybonds', '其他金融机构债', '否'),
        ('enterprisebonds', '企业债', '否'),
        ('enterprisebonds', '企业债', '是'),
        ('corporatebonds', '公司债', '否'),
        ('corporatebonds', '公司债', '是'),
        ('medium-termnotes', '中期票据', '否'),
        ('medium-termnotes', '中期票据', '是'),
        ('cps', '短期融资券', '否'),
        ('cps', '短期融资券', '是'),
        ('projectrevenuenotes', '项目收益票据', '否'),
        ('ppn', 'PPN', '否'),
        ('ppn', 'PPN', '是'),
        ('supranationalbonds', '国际机构债', '否'),
        ('agencybonds', '政府支持机构债', '否'),
        ('standardizednotes', '标准化票据', '否'),
        ('abs', 'ABS', '否'),
        ('convertiblebonds', '可转债', '否'),
        ('exchangeablebonds', '可交换债', '否'),
        ('detachableconvertiblebonds', '可分离转债', '否'),
    ]
    
    def __init__(self, engine, table_name='债券到期'):
        self.engine = engine
        self.table_name = table_name
        self.w = None
    
    def _init_wind(self):
        """初始化Wind API"""
        try:
            from WindPy import w
            self.w = w
            self.w.start()
            log_progress("Wind API初始化成功")
            return True
        except ImportError:
            log_progress("Wind API不可用")
            return False
    
    def _get_existing_dates(self, start_date: str) -> set:
        """获取已有数据的日期"""
        query = f"SELECT DISTINCT dt FROM {self.table_name} WHERE dt >= '{start_date}'"
        try:
            with self.engine.connect() as conn:
                dates = pd.read_sql(query, conn)
            return set(pd.to_datetime(dates['dt']).values)
        except Exception as e:
            log_progress(f"查询已有日期失败: {e}")
            return set()
    
    def _fetch_single_day_single_type(self, dt_str: str, bond_type_code: str, 
                                       bond_type_name: str, is_urban: str) -> Optional[pd.DataFrame]:
        """获取单个日期、单种债券类型的数据"""
        conceptbond = 'urbaninvestmentbonds(wind)' if is_urban == '是' else 'default'
        
        error_code, wsd_data = self.w.wset(
            "bondissuanceandmaturity",
            f"startdate={dt_str};enddate={dt_str};frequency=day;maingrade=all;zxgrade=all;"
            f"datetype=startdate;duedatetype=byactualpaymentdate;type=default;publishlimited=all;"
            f"bondtype={bond_type_code};dealmarket=allmarkets;conceptbond={conceptbond};"
            f"field=startdate,enddate,totalredemption",
            usedf=True
        )
        
        if error_code == 0 and wsd_data is not None and not wsd_data.empty:
            wsd_data['bondtype'] = bond_type_name
            wsd_data['isurbaninvestmentbonds'] = is_urban
            wsd_data.rename(columns={'startdate': 'dt'}, inplace=True)
            if 'enddate' in wsd_data.columns:
                wsd_data.drop(columns=['enddate'], inplace=True)
            return wsd_data
        return None
    
    def collect(self, days_ahead: int = 7) -> Dict[str, Any]:
        """
        执行债券到期数据采集
        
        Args:
            days_ahead: 获取未来几天的数据
            
        Returns:
            采集结果统计
        """
        result = {
            'success': False,
            'total_records': 0,
            'dates_processed': 0,
            'errors': []
        }
        
        # 初始化Wind
        if not self._init_wind():
            result['errors'].append('Wind API不可用')
            return result
        
        # 计算日期范围
        current_date = datetime.now()
        end_date = current_date + timedelta(days=days_ahead)
        dt0 = current_date.strftime('%Y-%m-%d')
        dt1 = end_date.strftime('%Y-%m-%d')
        
        log_progress(f"采集日期范围: {dt0} ~ {dt1}")
        
        # 获取已有日期
        existing_dates = self._get_existing_dates(dt0)
        log_progress(f"已有数据日期数: {len(existing_dates)}")
        
        # 循环处理每个日期
        for dt in pd.date_range(start=dt0, end=dt1):
            if dt in existing_dates:
                log_progress(f"跳过 {dt.strftime('%Y-%m-%d')} (已有数据)")
                continue
            
            dt_str = dt.strftime('%Y-%m-%d')
            log_progress(f"处理日期: {dt_str}")
            
            # 获取该日期所有债券类型的数据
            day_data = []
            for bond_type_code, bond_type_name, is_urban in self.BOND_TYPES:
                df = self._fetch_single_day_single_type(dt_str, bond_type_code, bond_type_name, is_urban)
                if df is not None:
                    day_data.append(df)
            
            if day_data:
                all_df = pd.concat(day_data, ignore_index=True)
                
                # 列名转换和入库
                column_mapping = generate_column_mapping(all_df.columns)
                change_column_names(self.table_name, column_mapping, self.engine, to_english=True)
                count = insert_data_to_db(all_df, self.table_name, self.engine)
                change_column_names(self.table_name, column_mapping, self.engine, to_english=False)
                
                result['total_records'] += count
            
            result['dates_processed'] += 1
        
        result['success'] = True
        log_progress(f"采集完成，共处理 {result['dates_processed']} 天，{result['total_records']} 条记录")
        return result


print("BondMaturityCollector类定义完成")

### 债券新发行数据采集类

In [ ]:
class BondIssueCollector:
    """债券新发行数据采集器"""
    
    # 同花顺债券类型代码
    IFIND_BOND_TYPES = '640,640001,640001001,640001002,640001003,640001004,640002,640002001,640002002,640002003,640002004,640003,640004,640004001,640004001001,640004001002,640004001003,640004002,640004002001,640004002002,640004002003,640004002004,640004002005,640004003,640004003001,640004003002,640004003003,640004003004,640004003005,640005,640005001,640005001001,640005001002,640005001003,640005002,640005002001,640005002002,640006,640007,640008,640008001,640008002,640008002001,640008002002,640008002003,640009,640009001,640009002,640009003,640009004,640010,640010001,640010001001,640010001002,640010001003,640010002,640010003,640010006,640011,640012,640013,640013001,640013002,640014,640014001,640014003,640015,640016,640016001,640016002,640018,640021,640021001,640021002,640022,640024'
    
    def __init__(self, engine, table_name='债券新发行1'):
        self.engine = engine
        self.table_name = table_name
        self.ifind = None
    
    def _init_ifind(self):
        """初始化同花顺iFinD API"""
        try:
            from iFinDPy import THS_iFinDLogin, THS_DR
            self.ifind = {'login': THS_iFinDLogin, 'dr': THS_DR}
            
            username = os.environ.get('IFIND_USERNAME', '')
            password = os.environ.get('IFIND_PASSWORD', '')
            
            if not username or not password:
                log_progress("iFinD用户名或密码未配置")
                return False
            
            login_result = self.ifind['login'](username, password)
            log_progress(f"iFinD登录结果: {login_result}")
            return login_result == 0 or login_result == -1  # -1表示已登录
        except ImportError:
            log_progress("iFinD API不可用")
            return False
    
    def _deduplicate(self):
        """执行去重"""
        with self.engine.begin() as connection:
            connection.execute(text("""
            CREATE TEMPORARY TABLE temp_table AS
            SELECT * FROM `{table}`
            WHERE sec_name IN (
                SELECT sec_name FROM `{table}` GROUP BY sec_name HAVING COUNT(*) = 1
            )
            UNION
            SELECT * FROM `{table}`
            WHERE trade_code IN (
                SELECT MIN(trade_code) FROM `{table}`
                WHERE trade_code NOT LIKE 'S%%' GROUP BY sec_name HAVING COUNT(*) > 1
            );
            DELETE FROM `{table}`;
            INSERT INTO `{table}` SELECT * FROM temp_table;
            DROP TEMPORARY TABLE temp_table;
            """.format(table=self.table_name)))
        log_progress("去重完成")
    
    def collect(self, days_start: int = 1, days_end: int = 30) -> Dict[str, Any]:
        """
        执行债券新发行数据采集
        
        Args:
            days_start: 开始日期偏移（相对于今天）
            days_end: 结束日期偏移（相对于今天）
            
        Returns:
            采集结果统计
        """
        result = {
            'success': False,
            'total_records': 0,
            'errors': []
        }
        
        # 初始化iFinD
        if not self._init_ifind():
            result['errors'].append('iFinD API不可用')
            return result
        
        # 计算日期范围
        current_date = datetime.now()
        start_date = current_date + timedelta(days=days_start)
        end_date = current_date + timedelta(days=days_end)
        dt0 = start_date.strftime('%Y%m%d')
        dt1 = end_date.strftime('%Y%m%d')
        
        log_progress(f"采集日期范围: {start_date.strftime('%Y-%m-%d')} ~ {end_date.strftime('%Y-%m-%d')}")
        
        # 调用iFinD API
        df = self.ifind['dr'](
            'p04524',
            f'sdate={dt0};edate={dt1};zqlx={self.IFIND_BOND_TYPES};'
            f'sclx=0;jglx=0;datetype=5;fxqx=0;ztpj=0;hy=0;qyxz=0;dq=0;gnfl=0',
            'jydm:Y,jydm_mc:Y,p04524_f005:Y,p04524_f006:Y,p04524_f009:Y,p04524_f029:Y,p04524_f063:Y',
            'format:dataframe'
        )
        
        if df is None or df.data is None:
            result['errors'].append('未获取到数据')
            return result
        
        df = df.data
        
        if df.empty:
            result['errors'].append('数据为空')
            return result
        
        # 重命名列
        df.columns = ['trade_code', 'sec_name', 'dt', 'planissueamount', 'bondterm', 'bondtype', 'isurbaninvestmentbonds']
        
        log_progress(f"获取数据: {len(df)}条")
        
        # 列名转换和入库
        column_mapping = generate_column_mapping(df.columns)
        change_column_names(self.table_name, column_mapping, self.engine, to_english=True)
        result['total_records'] = insert_data_to_db(df, self.table_name, self.engine)
        change_column_names(self.table_name, column_mapping, self.engine, to_english=False)
        
        # 执行去重
        self._deduplicate()
        
        result['success'] = True
        log_progress(f"采集完成，共 {result['total_records']} 条记录")
        return result


print("BondIssueCollector类定义完成")

### 主控制类

In [ ]:
class BondNewIssueSystem:
    """债券新发行系统主控制类"""
    
    def __init__(self):
        self.engine = get_db_engine()
        self.maturity_collector = BondMaturityCollector(self.engine)
        self.issue_collector = BondIssueCollector(self.engine)
    
    def run_maturity_collection(self, days_ahead: int = 7) -> Dict[str, Any]:
        """执行债券到期数据采集"""
        log_progress("="*50)
        log_progress("开始执行债券到期数据采集")
        log_progress("="*50)
        result = self.maturity_collector.collect(days_ahead)
        return result
    
    def run_issue_collection(self, days_start: int = 1, days_end: int = 30) -> Dict[str, Any]:
        """执行债券新发行数据采集"""
        log_progress("="*50)
        log_progress("开始执行债券新发行数据采集")
        log_progress("="*50)
        result = self.issue_collector.collect(days_start, days_end)
        return result
    
    def run_all(self) -> Dict[str, Any]:
        """执行全部采集任务"""
        log_progress("="*50)
        log_progress("开始执行全部采集任务")
        log_progress("="*50)
        
        results = {
            'maturity': self.run_maturity_collection(),
            'issue': self.run_issue_collection()
        }
        
        log_progress("="*50)
        log_progress("全部采集任务完成")
        log_progress("="*50)
        
        return results


print("BondNewIssueSystem类定义完成")

### 使用示例

In [ ]:
# 创建系统实例
# system = BondNewIssueSystem()

# 执行债券到期数据采集
# result_maturity = system.run_maturity_collection(days_ahead=7)

# 执行债券新发行数据采集
# result_issue = system.run_issue_collection(days_start=1, days_end=30)

# 执行全部采集任务
# results = system.run_all()

print("系统初始化代码已准备就绪")
print("要执行采集，请取消注释上面的代码并确保相关API可用")

---

### 本章小结

**已实现功能**:
- `BondMaturityCollector`: 债券到期数据采集器
- `BondIssueCollector`: 债券新发行数据采集器
- `BondNewIssueSystem`: 主控制系统
- 数据入库UPSERT逻辑
- 数据去重逻辑

**设计特点**:
- 面向对象设计，便于扩展和维护
- 统一的日志输出
- 错误处理和结果返回
- 配置与代码分离

---

**下一章**: [6_测试与验证](6_测试与验证.ipynb)